# MCP Architecture and Protocol

An LLM on its own only produces text. To read a ticket, query a database, or open a file, the application around the model has to connect it to those systems. Without a shared standard, if _M_ AI applications (such as IDE assistants or chat apps) each need to reach _N_ external systems (such as databases or APIs), one has to build _M × N_ custom connectors, each with its own data format, error handling, and security assumptions. The [Model Context Protocol](https://modelcontextprotocol.io/docs/getting-started/intro) (MCP) replaces that custom glue with one open protocol. Each system is wrapped once as an MCP server and each application implements an MCP client once, so the work grows as _M + N_ instead of _M × N_.

This notebook builds the model the rest of the repository relies on: who does what (roles), what a server can offer (primitives), and what actually travels between them (JSON-RPC messages). The examples are real messages exchanged with Northstar, the small operations server used in every notebook.

## Learning Objectives

By the end of this notebook, you should be able to:

- Distinguish host, client, server, and external-system responsibilities.
- Choose tools, resources, or prompts for a concrete workflow.
- Read the JSON-RPC requests, responses, and notifications a client and server exchange.
- Distinguish protocol errors from tool errors, and explain where validation and authorization belong.

## Roles: Host, Client, Server

Every MCP interaction involves the same four parties:

- **Host**: the AI application the user works in, such as VS Code or a desktop assistant. It runs the model, decides which servers to connect, and asks the user for approval.
- **Client**: a connector inside the host that speaks MCP to exactly one server. A host connected to three servers runs three clients.
- **Server**: a program that exposes capabilities through MCP. It can run locally as a subprocess or remotely behind an HTTP endpoint.
- **External system**: the filesystem, API, or database behind the server. The host never talks to it directly.

```mermaid
flowchart LR
    H["Host application"] --> C["MCP client<br/>(one per server)"]
    C -->|"discover, list,<br/>invoke or read"| S["Northstar MCP server"]
    S --> F["Local operation fixtures"]
```


The most important boundary is between model intent and protocol authority: the model can only *request* an action. The host and client decide which server capabilities the model sees and whether a request is actually sent.

## Primitives: Tools, Resources, Prompts

A server offers three kinds of capability, called primitives. The most useful question for choosing between them is *who decides when it is used*:

- **Tools:** functions the model calls when it decides it needs them. A tool may read data or change state; either way, the host controls access.
- **Resources:** read-only context, identified by a URI, that the host chooses to attach (e.g. a file the user selects).
- **Prompts:** reusable instruction templates the user picks explicitly, often as a slash command.

The Northstar server exposes one of each. Notebook 02 builds it with the official SDK.

| Primitive | Controlled by | Purpose | Northstar example |
|---|---|---|---|
| Tool | Model | Perform a controlled action | `create_escalation` |
| Resource | Application | Read approved context | `northstar://systems` |
| Prompt | User | Reuse a workflow instruction | `incident_update` |

Which primitive to use is a design decision, because the same data can often be exposed more than one way. Incident status, for example, could be a resource the host attaches to every conversation, or a tool the model calls with a ticket ID when it needs it. Choose based on who should decide when the data is used: the model, the application, or the user.

A server can also ask the host to collect user input through elicitation. Northstar does not use this client feature.

## The Protocol: JSON-RPC Messages

Roles and primitives describe *what exists*. The protocol describes *what is said*. MCP has two layers:

- The **data layer** defines the messages: [JSON-RPC 2.0](https://www.jsonrpc.org/specification) objects with a method name and parameters, such as `tools/list` or `tools/call`.
- The **transport layer** moves those messages: one JSON object per line over a subprocess's stdin and stdout (stdio) for local servers, or Streamable HTTP for remote ones. Notebook 04 compares them.

The data layer is the same on every transport, so everything below also applies to a remote server. JSON-RPC has three kinds of message:

| Kind | Has an `id`? | Gets a reply? | Example |
|---|---|---|---|
| Request | Yes | Yes, a response with the same `id` | `tools/call` |
| Response | Yes, copied from the request | No | A `result` on success, an `error` on failure |
| Notification | No | No | `notifications/cancelled` |

Every MCP request also carries a small envelope in `params._meta`: the protocol version (`2026-07-28`) and the client's capabilities. Because each request is self-contained, there is no session to set up first. To learn which versions and features a server supports, a client can send `server/discover`.

> **Note:** Older MCP versions start each connection with an `initialize` handshake instead. You will still see it in many tutorials, and the SDK in this repository supports both.

## A Complete Exchange

To escalate a ticket, the client can first ask the server what it supports, then lists the tools to learn their input schemas, and finally calls the one the model picks. The host and model only enter the picture between the last two requests, when the model decides to use the tool.

```mermaid
sequenceDiagram
    participant H as Host and model
    participant C as MCP client
    participant S as Northstar server
    C->>S: server/discover (optional)
    S-->>C: supported versions and capabilities
    C->>S: tools/list
    S-->>C: create_escalation and its input schema
    H->>C: model asks to escalate MCP-1002
    C->>S: tools/call create_escalation
    S-->>C: structured result
    C->>H: result becomes model context
```

The `tools/list` response describes each tool with a name, a description, and a JSON Schema for its input. That schema is how the client, and through it the model, learns which arguments a tool needs and what type each one must be, without reading the server's code:

```json
{
  "name": "create_escalation",
  "description": "Create an escalation plan for a support ticket.",
  "inputSchema": {
    "type": "object",
    "properties": {"ticket_id": {"type": "string"}},
    "required": ["ticket_id"]
  }
}
```

When the model decides to escalate, the client sends a `tools/call` request with the tool name, the arguments, and the `_meta` envelope:

```json
{
  "jsonrpc": "2.0",
  "id": 3,
  "method": "tools/call",
  "params": {
    "name": "create_escalation",
    "arguments": {"ticket_id": "MCP-1002"},
    "_meta": {
      "io.modelcontextprotocol/protocolVersion": "2026-07-28",
      "io.modelcontextprotocol/clientCapabilities": {}
    }
  }
}
```

The server replies with the same `id`. The result holds structured data that matches the tool's output schema (shortened here). The full result also repeats it as text in `content` for models and older clients:

```json
{
  "jsonrpc": "2.0",
  "id": 3,
  "result": {
    "resultType": "complete",
    "isError": false,
    "structuredContent": {
      "ticket_id": "MCP-1002",
      "policy_id": "policy-critical-incident",
      "external_system": "incident_db",
      "requires_human_approval": true
    }
  }
}
```

Note `requires_human_approval: true`: the server can flag that a critical ticket needs a human, but only the host can actually stop and ask.

## When Something Goes Wrong: Two Kinds Of Error

MCP separates failures by *who can fix them*:

- A **protocol error** means the request itself could not be handled: an unknown method, malformed parameters, a missing envelope. The response carries an `error` object with a numeric `code` instead of a `result`. These are usually bugs in the client or server that a model cannot fix.
- A **tool error** means the request was valid, but the tool failed: an unknown ticket, an API timeout, a value out of range. The response is a normal `result` with `isError: true` and an explanation in `content`. The client should pass it to the model, which can often correct itself and retry.

Here is what each looks like on the wire. A request for a method that does not exist, such as `tickets/delete`, gets a protocol error. `-32601` is the standard JSON-RPC code for an unknown method:

```json
{
  "jsonrpc": "2.0",
  "id": 4,
  "error": {"code": -32601, "message": "Method not found", "data": "tickets/delete"}
}
```

A valid `tools/call` for a ticket that does not exist, such as `MCP-9999`, reaches the tool, so it gets a tool error instead:

```json
{
  "jsonrpc": "2.0",
  "id": 5,
  "result": {
    "resultType": "complete",
    "isError": true,
    "content": [
      {
        "type": "text",
        "text": "Error executing tool create_escalation: No row found for ticket_id=MCP-9999"
      }
    ]
  }
}
```

The model can read this message and act on it, for example by asking the user to check the ticket ID.

## Where Validation And Authorization Belong

The tool error raises a design question: which layer should check what? Three checks are easy to mix up:

- **Validation** → Is this request well-formed? Is `ticket_id` present, and is it a string?
- **Authorization** → May this caller do this? May this user read `MCP-1002`?
- **Approval** → Does a human agree to this specific action, now? Should this critical incident be escalated?

| Layer | Responsibility | Northstar Example |
|---|---|---|
| Host | Decides which servers are connected and which tools the model sees. Asks the user to approve sensitive actions. | No interactive host is implemented. The plan flags critical tickets for review, but nothing pauses for approval. |
| Client | Calls discovered tools. Can check arguments against the `inputSchema` before sending. | The examples send missing arguments to demonstrate server-side validation; the client does not reject them locally. |
| Server | Validates every input and enforces access control, whatever the client already checked. | Rejects missing arguments and `MCP-9999`. Caller-level authorization would be required in production, but is not implemented here. |
| External system | Enforces its own permissions as the last line of defense. | JSON fixtures describe `read:incidents`; no real `incident_db` or scope enforcement is connected. |

Northstar only returns a plan; it does not execute an escalation. A production host could ask a person to review that plan, then require approval **before** any later sensitive action.

A schema only tells you that a request is well-formed. A valid request for a ticket the caller may not see must still be refused, and an authorized action can still need a human's approval. Notebook 04 covers how remote servers verify who is calling.

## Check Your Understanding

1. Can a tool be read-only?
2. Must a given piece of data always be exposed as a resource rather than a tool?
3. If a request passes a tool's input schema, does that mean the caller may access the data it asks for?
4. A `tools/call` response contains `"isError": true`. Did the protocol fail?
5. Why does every request carry the protocol version instead of agreeing on it once per connection?

<details>
<summary>Show Reference Answers</summary>

1. Yes. Tools can read data or change state; being a tool does not imply a write.
2. No. The same data can be a resource the application attaches as context, or a tool the model calls with parameters when it needs it. The server author chooses based on who should decide when it is used.
3. No. Schema validation only checks the shape of the input; authorization checks whether this caller may access that specific data.
4. No. The request was valid and the tool ran, but it failed. A protocol failure returns an `error` object instead of a `result`. The client should show the tool error to the model so it can correct itself.
5. Because each request is self-contained: the server can handle it on its own, without remembering earlier messages on the same connection. That lets a remote server spread requests across processes and lets clients send unrelated requests over one connection.

</details>

## Further Reading

- [MCP Architecture Overview](https://modelcontextprotocol.io/docs/learn/architecture): Host, client, and server responsibilities.
- [Understanding MCP Servers](https://modelcontextprotocol.io/docs/learn/server-concepts): Tools, resources, and prompts, and who controls each.
- [Base Protocol Overview](https://modelcontextprotocol.io/specification/2026-07-28/basic): JSON-RPC messages, error codes, and the `_meta` envelope.
- [Versioning and Compatibility](https://modelcontextprotocol.io/specification/2026-07-28/basic/versioning): Per-request protocol versions and the older `initialize` handshake.
- [MCP Tools](https://modelcontextprotocol.io/specification/2026-07-28/server/tools): Protocol errors, tool errors, and the security requirements on servers and clients.